# Day 4 - Feature Engineering and Ensembles (RetailMind AI)

**Goal:** add domain features, reduce redundancy with PCA, test bagging (Random Forest) and boosting (XGBoost), then finalise the production churn model.

> Run Day 3 first: this notebook compares against `models/churn_v3.joblib`.
> XGBoost is needed for one cell: `pip install xgboost`.

## 1. Setup

In [1]:
import sys
sys.path.append("..")

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

from src.config import MODEL_PATH
from src.data_loader import load_data, split_features_target
from src.preprocessing import build_preprocessor
from src.features import add_features
from src.evaluate import get_metrics

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 2. Same data and split as before

In [2]:
df = load_data()
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

Train rows: 5634 | Test rows: 1409


## 3. Engineer domain features
`add_features()` (in `src/features.py`) adds:
- **tenure_group:** 0-12, 13-24, 25-48, 49-72 months
- **avg_monthly_spend** and **charge_ratio:** how current charges compare with what the customer has paid on average
- **num_add_ons:** how many extra services they use
- **flags:** month-to-month contract, has internet, pays automatically, has security/support

In [3]:
X_fe = add_features(X_train)
print("Columns before:", X_train.shape[1], "| after:", X_fe.shape[1])
X_fe[["tenure", "tenure_group", "avg_monthly_spend", "charge_ratio",
      "num_add_ons", "is_month_to_month", "auto_pay", "has_support"]].head()

Columns before: 19 | after: 27


,tenure,tenure_group,avg_monthly_spend,charge_ratio,num_add_ons,is_month_to_month,auto_pay,has_support
3738,35,25-48,47.268056,1.019308,3,1,0,0
3151,15,13-24,71.971875,1.029164,1,1,0,1
4860,13,13-24,42.167857,0.939356,3,0,0,1
3867,26,25-48,70.581481,1.026802,4,0,1,0
3810,1,0-12,22.275000,1.914071,0,1,0,0


Do the new features actually relate to churn? Churn rate by group:

In [4]:
check = X_fe.assign(churned=y_train.values)
print(check.groupby("tenure_group")["churned"].mean().round(3))
print()
print(check.groupby("num_add_ons")["churned"].mean().round(3))

tenure_group
0-12     0.473
13-24    0.286
25-48    0.202
49-72    0.099
Name: churned, dtype: float64

num_add_ons
0    0.209
1    0.458
2    0.362
3    0.278
4    0.230
5    0.125
6    0.047
Name: churned, dtype: float64


New customers (0-12 months) churn at about 47%, falling to about 10% for customers with 49-72 months. For add-on services, churn drops steadily from about 46% (1 add-on) to about 5% (6 add-ons). Customers with 0 add-ons churn less (about 21%) because most of them have no internet service at all. So these features carry real signal.

## 4. Compare models
A helper builds the full pipeline: **features → preprocessing → (optional PCA) → model**. Everything is inside one pipeline, so the saved model accepts a plain customer record.

In [5]:
def make_pipe(model, use_pca=False):
    steps = [
        ("features", FunctionTransformer(add_features)),
        ("preprocess", build_preprocessor(X_fe)),
    ]
    if use_pca:
        steps.append(("pca", PCA(n_components=0.95, random_state=42)))
    steps.append(("model", model))
    return Pipeline(steps)

results = {}
fitted = {}

# Day 3 champion (tuned Random Forest, original features) for comparison
day3_model = joblib.load(MODEL_PATH.parent / "churn_v3.joblib")
day3_name = "Day 3 champion (tuned RF)"
results[day3_name] = get_metrics(day3_model, X_test, y_test)
fitted[day3_name] = day3_model

pd.DataFrame(results).T

,Accuracy,Precision,Recall,F1,ROC-AUC
Day 3 champion (tuned RF),0.7559,0.5268,0.7888,0.6317,0.8432


### Bagging: Random Forest with engineered features
Same tuned settings as Day 3, now with the new features.

In [6]:
def new_rf():
    return RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=10,
                                  class_weight="balanced", random_state=42)

rf_fe = make_pipe(new_rf())
rf_fe.fit(X_train, y_train)
fitted["RF + features"] = rf_fe
results["RF + features"] = get_metrics(rf_fe, X_test, y_test)
pd.DataFrame(results).T

,Accuracy,Precision,Recall,F1,ROC-AUC
Day 3 champion (tuned RF),0.7559,0.5268,0.7888,0.6317,0.8432
RF + features,0.7594,0.5318,0.7834,0.6335,0.8447


### PCA: fewer, less redundant features
One-hot encoding creates many overlapping columns. PCA combines them into fewer columns while keeping 95% of the information.

In [7]:
encoded = build_preprocessor(X_fe).fit_transform(X_fe)
pca_check = PCA(n_components=0.95, random_state=42).fit(encoded)
print("Features before PCA:", encoded.shape[1], "-> after PCA:", pca_check.n_components_)

rf_pca = make_pipe(new_rf(), use_pca=True)
rf_pca.fit(X_train, y_train)
fitted["RF + features + PCA"] = rf_pca
results["RF + features + PCA"] = get_metrics(rf_pca, X_test, y_test)
pd.DataFrame(results).T

Features before PCA: 56 -> after PCA: 20


,Accuracy,Precision,Recall,F1,ROC-AUC
Day 3 champion (tuned RF),0.7559,0.5268,0.7888,0.6317,0.8432
RF + features,0.7594,0.5318,0.7834,0.6335,0.8447
RF + features + PCA,0.7530,0.5242,0.7540,0.6184,0.8373


### Boosting: XGBoost with engineered features
Boosting builds trees one after another, each fixing the previous one's mistakes. `scale_pos_weight` plays the same role as `class_weight="balanced"`.

Install first if needed: `pip install xgboost`

In [ ]:
from xgboost import XGBClassifier

ratio = (y_train == 0).sum() / (y_train == 1).sum()   # stayers per churner

xgb = make_pipe(XGBClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.1,
    scale_pos_weight=ratio, eval_metric="logloss", random_state=42,
))
xgb.fit(X_train, y_train)
fitted["XGBoost + features"] = xgb
results["XGBoost + features"] = get_metrics(xgb, X_test, y_test)
pd.DataFrame(results).T

## 5. Final comparison and champion

In [8]:
comparison = pd.DataFrame(results).T
comparison.sort_values("ROC-AUC", ascending=False)

,Accuracy,Precision,Recall,F1,ROC-AUC
RF + features,0.7594,0.5318,0.7834,0.6335,0.8447
Day 3 champion (tuned RF),0.7559,0.5268,0.7888,0.6317,0.8432
RF + features + PCA,0.7530,0.5242,0.7540,0.6184,0.8373


In [9]:
final_name = comparison["ROC-AUC"].idxmax()
print("Final model (best ROC-AUC):", final_name)
comparison.loc[[final_name]]

Final model (best ROC-AUC): RF + features


,Accuracy,Precision,Recall,F1,ROC-AUC
RF + features,0.7594,0.5318,0.7834,0.6335,0.8447


**Why this one:** in this run, Random Forest with the engineered features had the best ROC-AUC on the hold-out set while keeping recall high (about 0.78). ROC-AUC does not depend on the decision threshold, so it is a fairer way to rank models; we then check recall and F1.

**What we learned**
- The engineered features gave only a **small** gain over the Day 3 champion (ROC-AUC 0.8432 to 0.8447). The original columns already contain most of the signal.
- **PCA** cut the features from 56 to 20, but it lowered recall (0.78 to 0.75) and ROC-AUC, so we don't use it in the final model. It could still be useful if speed or model size ever matters.

## 6. Finalise: save the production model
The saved pipeline contains **feature engineering + preprocessing + model**, so the preprocessing is frozen together with the model.

In [10]:
final_model = fitted[final_name]
joblib.dump(final_model, MODEL_PATH.parent / "churn_v4.joblib")
print("Saved churn_v4.joblib")

Saved churn_v4.joblib


Test the saved model through the `predict()` service with a raw customer record:

In [11]:
from src.predict import predict

new_customer = {
    "gender": "Female", "SeniorCitizen": 0, "Partner": "No", "Dependents": "No",
    "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "No",
    "StreamingMovies": "No", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 70.35, "TotalCharges": 140.7,
}
predict(new_customer, model_path=MODEL_PATH.parent / "churn_v4.joblib")

{'churn_probability': 0.8228, 'churn_prediction': 'Yes', 'risk_level': 'High'}

## Summary (Phase 1 deliverable)
Update the README baseline table with every day's best model (Day 1 to Day 4). Phase 2 starts next with deep learning (a Keras ANN) benchmarked against this model.